# День 2 — Очистка текста и подготовка признаков

**Цель:** единый переиспользуемый препроцессинг для train и inference.

Скриптовый аналог: [`preprocess.py`](../preprocess.py).

In [1]:
import sys, os
# Добавляем корень проекта в путь, чтобы импортировать модули (config, preprocess, ...)
sys.path.insert(0, os.path.abspath('..'))

In [2]:
from preprocess import clean_text, load_and_preprocess

## 1. `clean_text` на примерах

Нижний регистр, удаление html/url, замена тикеров и чисел на плейсхолдеры, схлопывание пробелов.

In [3]:
samples = [
    '$ESI on lows, down $1.50 to $2.50 BK a real possibility',
    'Net sales <b>ROSE</b> 12% to EUR 131m   from EUR76m.',
    'Market remains stable despite volatility http://news.example.com',
]
for s in samples:
    print('IN :', s)
    print('OUT:', clean_text(s), '\n')

IN : $ESI on lows, down $1.50 to $2.50 BK a real possibility
OUT: ticker on lows, down $ number to $ number bk a real possibility 

IN : Net sales <b>ROSE</b> 12% to EUR 131m   from EUR76m.
OUT: net sales rose number % to eur 131m from eur76m. 

IN : Market remains stable despite volatility http://news.example.com
OUT: market remains stable despite volatility 



## 2. Полный препроцессинг датасета

In [4]:
df = load_and_preprocess()
print('Строк:', len(df))
print('Колонки:', list(df.columns))
df[['text', 'text_clean', 'label', 'word_count_clean', 'char_count', 'dollar_count']].head()

Строк: 5322
Колонки: ['text', 'sentiment', 'text_clean', 'label', 'word_count_clean', 'char_count', 'dollar_count']


,text,text_clean,label,word_count_clean,char_count,dollar_count
0,The GeoSolutions technology will leverage Bene...,the geosolutions technology will leverage bene...,2,32,218,0
1,"$ESI on lows, down $1.50 to $2.50 BK a real po...","ticker on lows, down $ number to $ number bk a...",0,13,63,3
2,"For the last quarter of 2010 , Componenta 's n...","for the last quarter of number , componenta 's...",2,39,195,0
3,According to the Finnish-Russian Chamber of Co...,according to the finnish-russian chamber of co...,1,20,128,0
4,The Swedish buyout firm has sold its remaining...,the swedish buyout firm has sold its remaining...,1,23,137,0


## 3. Проверка меток

`negative→0, neutral→1, positive→2`

In [5]:
df['label'].value_counts().sort_index()

label
0     592
1    2878
2    1852
Name: count, dtype: int64

## Вывод
Препроцессинг детерминирован и вынесен в отдельный модуль — те же шаги применяются и при обучении, и на инференсе, что гарантирует согласованность.